# Capítulo 14. Comparación integral y selección de modelos

Este cuaderno compara **Regresión Logística, k-NN, Árbol de decisión, Random Forest, SVM, Naive Bayes y Red Neuronal** con la misma muestra, la misma partición y las mismas métricas.

> **Uso académico:** los datos COVID-19 se utilizan para aprender metodología de clasificación. Los resultados no son un sistema clínico ni estimaciones de riesgo individual.

El procedimiento mantiene tres conjuntos: **entrenamiento**, **validación** y **prueba**. La prueba se consulta únicamente al final.

In [ ]:
# Preparación automática del entorno R
paquetes <- c('readr','dplyr','tidyr','class','rpart','ranger','e1071','neuralnet')
faltantes <- paquetes[!vapply(paquetes, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes) > 0) install.packages(faltantes, repos='https://cloud.r-project.org')
invisible(lapply(paquetes, library, character.only = TRUE))

dir.create('datos/covid19/procesados', recursive = TRUE, showWarnings = FALSE)
url_base <- 'https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r/edicion-2026-bilingue/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz'
ruta <- 'datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz'
if (!file.exists(ruta)) download.file(url_base, ruta, mode='wb', quiet=TRUE)
cat('Base preparada:', ruta, '\n')

## 1. Crear un benchmark común

Para que la comparación sea justa, todos los modelos usarán las mismas siete variables y exactamente las mismas observaciones. Se toma una muestra reproducible para mantener razonable el tiempo de ejecución en Colab.

In [ ]:
covid <- readr::read_csv(ruta, show_col_types = FALSE) |>
  dplyr::select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION, OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
  tidyr::drop_na() |>
  dplyr::mutate(clase = factor(MURIO, levels=c(0,1), labels=c('Sin defunción','Defunción'))) |>
  dplyr::select(-MURIO)

set.seed(2026)
n_por_clase <- min(2500, min(table(covid$clase)))
benchmark <- covid |>
  dplyr::group_by(clase) |>
  dplyr::slice_sample(n = n_por_clase) |>
  dplyr::ungroup()

table(benchmark$clase)

## 2. Partición entrenamiento-validación-prueba

Primero se reserva 20 % para prueba. Del 80 % restante se usa 75 % para entrenamiento y 25 % para validación, dando aproximadamente 60/20/20.

In [ ]:
muestreo_estratificado <- function(y, proporcion, semilla) {
  set.seed(semilla)
  unlist(lapply(split(seq_along(y), y), function(i) sample(i, floor(proporcion * length(i)))))
}

idx_dev <- muestreo_estratificado(benchmark$clase, 0.80, 2026)
desarrollo <- benchmark[idx_dev, ]
prueba <- benchmark[-idx_dev, ]
idx_train <- muestreo_estratificado(desarrollo$clase, 0.75, 2027)
entrenamiento <- desarrollo[idx_train, ]
validacion <- desarrollo[-idx_train, ]

rbind(
  entrenamiento = prop.table(table(entrenamiento$clase)),
  validacion = prop.table(table(validacion$clase)),
  prueba = prop.table(table(prueba$clase))
)

In [ ]:
metricas <- function(real, predicho) {
  niveles <- c('Sin defunción','Defunción')
  real <- factor(real, levels=niveles); predicho <- factor(predicho, levels=niveles)
  m <- table(Real=real, Predicho=predicho)
  VP <- m['Defunción','Defunción']; FN <- m['Defunción','Sin defunción']
  FP <- m['Sin defunción','Defunción']; VN <- m['Sin defunción','Sin defunción']
  div <- function(a,b) if (b==0) NA_real_ else as.numeric(a/b)
  sens <- div(VP,VP+FN); esp <- div(VN,VN+FP); prec <- div(VP,VP+FP)
  f1 <- if (is.na(prec)||is.na(sens)||prec+sens==0) NA_real_ else 2*prec*sens/(prec+sens)
  data.frame(exactitud=div(VP+VN,sum(m)), sensibilidad=sens, especificidad=esp,
             exactitud_balanceada=mean(c(sens,esp),na.rm=TRUE), precision=prec, f1=f1)
}

seleccionar_umbral <- function(prob, real) {
  candidatos <- seq(0.20,0.80,by=0.05)
  tab <- do.call(rbind,lapply(candidatos,function(u) {
    p <- factor(ifelse(prob>=u,'Defunción','Sin defunción'), levels=c('Sin defunción','Defunción'))
    cbind(umbral=u, metricas(real,p))
  }))
  tab$umbral[which.max(tab$exactitud_balanceada)]
}

## 3. Regresión logística: seleccionar umbral en validación

In [ ]:
modelo_log_val <- glm(clase ~ ., data=entrenamiento, family=binomial)
prob_log_val <- predict(modelo_log_val, validacion, type='response')
umbral_log <- seleccionar_umbral(prob_log_val, validacion$clase)
umbral_log

## 4. k-NN: seleccionar k en validación

In [ ]:
predictores <- setdiff(names(entrenamiento),'clase')
Xtr <- as.matrix(entrenamiento[predictores]); Xva <- as.matrix(validacion[predictores])
medias <- colMeans(Xtr); desv <- apply(Xtr,2,sd); desv[desv==0] <- 1
Xtr_z <- scale(Xtr,medias,desv); Xva_z <- scale(Xva,medias,desv)
ks <- c(3,5,11,21)
tabla_knn <- do.call(rbind,lapply(ks,function(k) {
  p <- class::knn(Xtr_z,Xva_z,entrenamiento$clase,k=k)
  cbind(k=k,metricas(validacion$clase,p))
}))
tabla_knn
k_opt <- tabla_knn$k[which.max(tabla_knn$exactitud_balanceada)]

## 5. Árbol de decisión: seleccionar complejidad

In [ ]:
arbol_grande <- rpart::rpart(clase ~ ., data=entrenamiento, method='class',
  control=rpart::rpart.control(cp=0, minsplit=30, minbucket=10, maxdepth=8, xval=5))
cp_opt <- arbol_grande$cptable[which.min(arbol_grande$cptable[,'xerror']),'CP']
arbol_val <- prune(arbol_grande, cp=cp_opt)
metricas(validacion$clase,predict(arbol_val,validacion,type='class'))

## 6. Random Forest: seleccionar mtry en validación

In [ ]:
mtries <- c(2,3,4)
tabla_rf <- do.call(rbind,lapply(mtries,function(m) {
  mod <- ranger::ranger(clase ~ ., data=entrenamiento, num.trees=300, mtry=m,
                         min.node.size=20, seed=2026, classification=TRUE)
  p <- predict(mod,validacion)$predictions
  cbind(mtry=m,metricas(validacion$clase,p))
}))
tabla_rf
mtry_opt <- tabla_rf$mtry[which.max(tabla_rf$exactitud_balanceada)]

## 7. SVM radial: seleccionar cost y gamma

In [ ]:
grid_svm <- expand.grid(cost=c(0.5,1,2), gamma=c(0.05,0.1))
tabla_svm <- do.call(rbind,lapply(seq_len(nrow(grid_svm)),function(i) {
  g <- grid_svm[i,]
  mod <- e1071::svm(clase ~ ., data=entrenamiento, kernel='radial',
                    cost=g$cost, gamma=g$gamma, scale=TRUE)
  p <- predict(mod,validacion)
  cbind(g,metricas(validacion$clase,p))
}))
tabla_svm
fila_svm <- which.max(tabla_svm$exactitud_balanceada)
cost_opt <- tabla_svm$cost[fila_svm]; gamma_opt <- tabla_svm$gamma[fila_svm]

## 8. Naive Bayes: seleccionar suavizado de Laplace

In [ ]:
laps <- c(0,0.5,1,2)
tabla_nb <- do.call(rbind,lapply(laps,function(l) {
  mod <- e1071::naiveBayes(clase ~ ., data=entrenamiento, laplace=l)
  p <- predict(mod,validacion,type='class')
  cbind(laplace=l,metricas(validacion$clase,p))
}))
tabla_nb
lap_opt <- tabla_nb$laplace[which.max(tabla_nb$exactitud_balanceada)]

## 9. Red neuronal: seleccionar arquitectura en validación

In [ ]:
# Para neuralnet estandarizamos EDAD y NUM_COMORBILIDADES; las binarias quedan 0/1.
continuas <- c('EDAD','NUM_COMORBILIDADES')
prep_nn <- function(train, other) {
  m <- sapply(train[continuas],mean); s <- sapply(train[continuas],sd); s[s==0] <- 1
  a <- train; b <- other
  a[continuas] <- scale(a[continuas],m,s); b[continuas] <- scale(b[continuas],m,s)
  a$y <- ifelse(a$clase=='Defunción',1,0); b$y <- ifelse(b$clase=='Defunción',1,0)
  a$clase <- NULL; b$clase <- NULL
  list(train=a, other=b)
}
nn_val <- prep_nn(entrenamiento,validacion)
arquitecturas <- list(c(3),c(5),c(5,3))
resultados_nn <- vector('list',length(arquitecturas))
modelos_nn <- vector('list',length(arquitecturas))
for (i in seq_along(arquitecturas)) {
  set.seed(4000+i)
  mod <- neuralnet::neuralnet(y ~ EDAD+NEUMONIA+DIABETES+HIPERTENSION+OBESIDAD+RENAL_CRONICA+NUM_COMORBILIDADES,
    data=nn_val$train, hidden=arquitecturas[[i]], linear.output=FALSE, lifesign='none', stepmax=5e5)
  prob <- as.numeric(neuralnet::compute(mod,nn_val$other[setdiff(names(nn_val$other),'y')])$net.result[,1])
  u <- seleccionar_umbral(prob,validacion$clase)
  p <- factor(ifelse(prob>=u,'Defunción','Sin defunción'),levels=c('Sin defunción','Defunción'))
  resultados_nn[[i]] <- cbind(arquitectura=paste(arquitecturas[[i]],collapse='-'),umbral=u,metricas(validacion$clase,p))
  modelos_nn[[i]] <- mod
}
tabla_nn <- do.call(rbind,resultados_nn)
tabla_nn
fila_nn <- which.max(tabla_nn$exactitud_balanceada)
arquitectura_opt <- arquitecturas[[fila_nn]]; umbral_nn <- as.numeric(tabla_nn$umbral[fila_nn])

## 10. Reentrenar con desarrollo y evaluar una sola vez en prueba

A partir de este punto ya no se cambian hiperparámetros. Los siete modelos se reentrenan con `desarrollo` y se evalúan sobre la misma `prueba`.

In [ ]:
# Regresión logística
log_final <- glm(clase ~ ., data=desarrollo, family=binomial)
prob_log <- predict(log_final,prueba,type='response')
pred_log <- factor(ifelse(prob_log>=umbral_log,'Defunción','Sin defunción'),levels=levels(desarrollo$clase))

# k-NN
Xdev <- as.matrix(desarrollo[predictores]); Xtest <- as.matrix(prueba[predictores])
mdev <- colMeans(Xdev); sdev <- apply(Xdev,2,sd); sdev[sdev==0] <- 1
pred_knn <- class::knn(scale(Xdev,mdev,sdev),scale(Xtest,mdev,sdev),desarrollo$clase,k=k_opt)

# Árbol
tree_big <- rpart::rpart(clase ~ .,data=desarrollo,method='class',control=rpart::rpart.control(cp=0,minsplit=30,minbucket=10,maxdepth=8,xval=5))
cp_final <- tree_big$cptable[which.min(tree_big$cptable[,'xerror']),'CP']
tree_final <- prune(tree_big,cp=cp_final)
pred_tree <- predict(tree_final,prueba,type='class')

# Random Forest
rf_final <- ranger::ranger(clase ~ .,data=desarrollo,num.trees=500,mtry=mtry_opt,min.node.size=20,seed=2026,classification=TRUE)
pred_rf <- predict(rf_final,prueba)$predictions

# SVM
svm_final <- e1071::svm(clase ~ .,data=desarrollo,kernel='radial',cost=cost_opt,gamma=gamma_opt,scale=TRUE)
pred_svm <- predict(svm_final,prueba)

# Naive Bayes
nb_final <- e1071::naiveBayes(clase ~ .,data=desarrollo,laplace=lap_opt)
pred_nb <- predict(nb_final,prueba,type='class')

# Red neuronal
nn_final_data <- prep_nn(desarrollo,prueba)
set.seed(2026)
nn_final <- neuralnet::neuralnet(y ~ EDAD+NEUMONIA+DIABETES+HIPERTENSION+OBESIDAD+RENAL_CRONICA+NUM_COMORBILIDADES,
  data=nn_final_data$train,hidden=arquitectura_opt,linear.output=FALSE,lifesign='none',stepmax=5e5)
prob_nn <- as.numeric(neuralnet::compute(nn_final,nn_final_data$other[setdiff(names(nn_final_data$other),'y')])$net.result[,1])
pred_nn <- factor(ifelse(prob_nn>=umbral_nn,'Defunción','Sin defunción'),levels=levels(desarrollo$clase))

In [ ]:
tabla_final <- dplyr::bind_rows(
  cbind(modelo='Regresión logística',metricas(prueba$clase,pred_log)),
  cbind(modelo=paste0('k-NN (k=',k_opt,')'),metricas(prueba$clase,pred_knn)),
  cbind(modelo='Árbol de decisión',metricas(prueba$clase,pred_tree)),
  cbind(modelo=paste0('Random Forest (mtry=',mtry_opt,')'),metricas(prueba$clase,pred_rf)),
  cbind(modelo='SVM radial',metricas(prueba$clase,pred_svm)),
  cbind(modelo=paste0('Naive Bayes (Laplace=',lap_opt,')'),metricas(prueba$clase,pred_nb)),
  cbind(modelo=paste0('Red neuronal (',paste(arquitectura_opt,collapse='-'),')'),metricas(prueba$clase,pred_nn))
)
tabla_final |> dplyr::arrange(dplyr::desc(exactitud_balanceada))

## Interpretación final

No elijas un modelo únicamente por la mayor exactitud. Compara sensibilidad, especificidad, exactitud balanceada, F1, estabilidad, tiempo de cómputo e interpretabilidad. Si dos modelos tienen resultados muy parecidos, suele ser razonable preferir el más simple y fácil de mantener.

**k-means no aparece en esta tabla** porque es un algoritmo no supervisado: sus clusters no son predicciones de una clase conocida.